# Week 5 · One prompt, many models

**GLBL 5053 · about 60 minutes · work with a partner, but use your own key**

A chat window hides two things: the request your words are wrapped in, and the fact that there are hundreds of models you could have sent it to. In this notebook you will send requests yourself, then put the same prompt to a ladder of models, from frontier models down to ones small enough to run on a phone.

By the end, you should be able to:

1. keep an API key out of code and notebook output;
2. point to the parts of a request and a response;
3. describe, from your own runs, what changes between a frontier model and a tiny one, and what does not; and
4. estimate what a task would cost at scale.

You do not need to write code. You will run cells, edit the prompts, and record what you see.

<sub>Adapted from the CPSC 1710 Lab 4 API activity; sections 3–9 are new for this course. Drafted by Claude (Anthropic) and Codex (OpenAI), based on the direction of Xiuye Chen.</sub>

## 0. Before you run anything: protect the key

An API key is a password that spends your course credit. A usable OpenRouter key begins with `sk-or-`. A key beginning with `ytf` only manages the Yale SOM Token Fund website and cannot call models.

**Never put a key in:**

- a line of code such as `api_key = "sk-or-..."`;
- a prompt to ChatGPT, Claude, Codex, or another assistant;
- a screenshot, discussion post, or shared document;
- a Git commit or GitHub repository; or
- notebook output.

If a key is exposed, disable it, create a replacement, and tell us. The credit is capped, so a mistake is recoverable.

### Add the key as a Colab secret

1. In Colab, click the **key icon** in the left sidebar.
2. Add a secret named exactly `OPENROUTER_API_KEY`.
3. Paste your `sk-or-...` key as its value.
4. Turn on **Notebook access** for that secret.

The setup cell asks Colab for the secret and never prints it. If you are running locally, set an environment variable with the same name before starting Jupyter.

In [ ]:
%pip -q install requests pandas numpy

In [ ]:
import os, json, time, random, re, datetime
from concurrent.futures import ThreadPoolExecutor

import requests
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("Add OPENROUTER_API_KEY as a Colab secret, then run again.")
if not OPENROUTER_API_KEY.startswith("sk-or-"):
    raise ValueError("This does not look like an OpenRouter key. Did you copy a ytf management key?")

BASE_URL = "https://openrouter.ai/api/v1"
HEADERS = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
pd.set_option("display.max_colwidth", 80)
print("Ready. The key was loaded but not displayed.")

## 1. Ask the API about your key

An API is a set of web addresses that accept structured requests. Here we send a `GET` request to the address ending in `/key`. The server reads the secret header to work out which key is asking, and returns data about it.

In [ ]:
key_response = requests.get(f"{BASE_URL}/key", headers=HEADERS, timeout=30)
key_response.raise_for_status()
key_info = key_response.json()["data"]

print("Spending limit:   $", key_info.get("limit"), sep="")
print("Already used:     $", key_info.get("usage"), sep="")
print("Remaining:        $", key_info.get("limit_remaining"), sep="")
print("Expiration (UTC):", key_info.get("expires_at"))
START_USAGE = key_info.get("usage") or 0

**Checkpoint 1.** Which part of that request proves who you are? Which part names what you want? Write the *Already used* number in your log; you will compare it at the end.

## 2. One request, taken apart

Before running the next cell, find the four ingredients of a chat request in the code:

1. **destination** — the web address the request is sent to;
2. **authorization** — the header carrying your key;
3. **model** — an ID that says which model should answer; and
4. **input** — a list of messages, each with a role and some content.

The cell prints exactly what is sent (minus the key) and exactly what comes back.

In [ ]:
request_body = {
    "model": "openai/gpt-6-luna",
    "messages": [
        {"role": "system", "content": "Be concrete and concise."},
        {"role": "user", "content": "In two sentences, what is a language model?"},
    ],
    "max_tokens": 1000,
}

print("SENT TO:", f"{BASE_URL}/chat/completions")
print(json.dumps(request_body, indent=2))

raw = requests.post(f"{BASE_URL}/chat/completions", headers=HEADERS, json=request_body, timeout=120).json()

def trimmed(x):
    # Shorten very long strings and drop bulky internals so the response is readable.
    if isinstance(x, dict):
        return {k: trimmed(v) for k, v in x.items() if k != "reasoning_details"}
    if isinstance(x, list):
        return [trimmed(v) for v in x]
    if isinstance(x, str) and len(x) > 400:
        return x[:400] + " ..."
    return x

print("\nCAME BACK:")
print(json.dumps(trimmed(raw), indent=2))

**Checkpoint 2.** In the response, find:

- the answer itself (`choices` → `message` → `content`);
- how many tokens went in and came out (`usage`); and
- what this one request cost, in dollars (`usage` → `cost`).

A chat app shows you only the first of these. Everything else in this notebook is this same request, repeated with a different `model` line.

## 3. The catalog, and a ladder through it

OpenRouter is a broker: one key, one request format, and it forwards your request to whichever company runs the model you name. The next cell downloads its public catalog and picks out nine models that span the price range, in five rough tiers.

The tiers here are **price tiers**, not a quality ranking. Whether price tracks quality is the thing you are about to test. For a guide to reading the catalog, see the [model overview page](https://glbl5053.github.io/labs/week-05/models.html).

In [ ]:
CATALOG = {m["id"]: m for m in requests.get(f"{BASE_URL}/models", timeout=60).json()["data"]}
print(len(CATALOG), "models in the catalog today, from",
      len({i.split("/")[0].lstrip("~") for i in CATALOG}), "organizations.")

LADDER = {
    "anthropic/claude-fable-5.1":       "1 frontier",
    "openai/gpt-6-astra":               "1 frontier",
    "anthropic/claude-sonnet-5.5":      "2 workhorse",
    "google/gemini-3.8-flash":          "2 workhorse",
    "deepseek/deepseek-v4.1-flash":     "3 fast and cheap",
    "openai/gpt-6-luna":                "3 fast and cheap",
    "google/gemma-4-31b-it":            "4 small, open weights",
    "mistralai/ministral-3b-2512":      "5 tiny",
    "meta-llama/llama-3.2-1b-instruct": "5 tiny",
}

# Other models you can swap in later. Copy an ID into LADDER, or use it in section 9.
EXTRAS = [
    "anthropic/claude-opus-5.5", "openai/gpt-6.1-sol", "x-ai/grok-4.7", "meta/muse-spark-1.3",
    "qwen/qwen3.8-max-0902", "z-ai/glm-5.3-flash", "moonshotai/kimi-k3", "anthropic/claude-haiku-4.5",
    "ibm-granite/granite-4.2-8b", "meta-llama/llama-3.2-3b-instruct", "liquid/lfm-2.5-2.6b:free",
]

missing = [m for m in LADDER if m not in CATALOG]
for m in missing:
    print("Not in today's catalog, skipping:", m)
    del LADDER[m]

def price(model, side):
    # Dollars per million tokens. side is "prompt" (input) or "completion" (output).
    return float(CATALOG[model]["pricing"].get(side, 0)) * 1_000_000

def ladder_table(models=None):
    rows = []
    for m in (models or LADDER):
        c = CATALOG[m]
        rows.append({
            "tier": LADDER.get(m, ""),
            "model": m,
            "released": datetime.datetime.fromtimestamp(c["created"]).strftime("%b %Y"),
            "context (tokens)": f'{c["context_length"]:,}',
            "$ per M in": round(price(m, "prompt"), 3),
            "$ per M out": round(price(m, "completion"), 3),
        })
    return pd.DataFrame(rows)

display(ladder_table())
top, bottom = max(LADDER, key=lambda m: price(m, "completion")), min(LADDER, key=lambda m: price(m, "completion"))
print(f"Output from {top} costs {price(top, 'completion') / price(bottom, 'completion'):,.0f}x "
      f"what output from {bottom} costs.")

**Checkpoint 3 — predict before you test.** Prices are in dollars per *million* tokens, and a token is roughly three-quarters of a word. Before going on, write down:

- For a simple question, which rung do you expect to be the lowest one that still gives a usable answer?
- What kind of question do you expect to separate the top of the ladder from the bottom?

The next cell defines the helper functions used in the rest of the notebook. **You do not need to read it. Run it and move on.**

In [ ]:
LOG = []   # every request made in this session, so we can total the bill

def ask(model, prompt, system=None, max_tokens=2000):
    # Send one chat request. Returns the answer plus the receipts: time, tokens, cost.
    messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    body = {"model": model, "messages": messages, "max_tokens": max_tokens}
    if "reasoning" in CATALOG.get(model, {}).get("supported_parameters", []):
        body["reasoning"] = {"effort": "low"}   # models that "think" first: keep it short
    row = {"tier": LADDER.get(model, ""), "model": model, "answer": "", "seconds": None,
           "tokens in": None, "tokens out": None, "of which thinking": None, "cost $": 0.0, "note": ""}
    start = time.time()
    try:
        for attempt in range(2):
            r = requests.post(f"{BASE_URL}/chat/completions", headers=HEADERS, json=body, timeout=240)
            if r.status_code in (429, 500, 502, 503) and attempt == 0:
                time.sleep(4)   # busy or briefly unavailable: wait and try once more
                continue
            break
        data = r.json()
        if "error" in data:
            raise RuntimeError(str(data["error"].get("message", data["error"]))[:160])
        choice, usage = data["choices"][0], data.get("usage") or {}
        if choice.get("error"):
            raise RuntimeError(str(choice["error"].get("message", choice["error"]))[:160])
        content = choice["message"].get("content") or ""
        if isinstance(content, list):
            content = " ".join(part.get("text", "") for part in content if isinstance(part, dict))
        row["answer"] = content.strip()
        row["tokens in"] = usage.get("prompt_tokens")
        row["tokens out"] = usage.get("completion_tokens")
        row["of which thinking"] = (usage.get("completion_tokens_details") or {}).get("reasoning_tokens") or 0
        cost = usage.get("cost")
        if cost is None and model in CATALOG:   # no bill attached: estimate from list prices
            cost = ((row["tokens in"] or 0) * price(model, "prompt")
                    + (row["tokens out"] or 0) * price(model, "completion")) / 1_000_000
        row["cost $"] = float(cost or 0)
        if not row["answer"]:
            row["note"] = "no visible answer (stopped: " + str(choice.get("finish_reason")) + ")"
    except Exception as e:
        row["note"] = "ERROR: " + str(e)[:160]
    row["seconds"] = round(time.time() - start, 1)
    LOG.append(row)
    return row

def compare(prompt, models=None, system=None, max_tokens=2000):
    # Send the same prompt to several models at once. Results come back in ladder order.
    models = list(models or LADDER)
    with ThreadPoolExecutor(max_workers=8) as pool:
        return list(pool.map(lambda m: ask(m, prompt, system, max_tokens), models))

def receipts(results):
    # The numbers, without the answers.
    cols = ["tier", "model", "seconds", "tokens in", "tokens out", "of which thinking", "cost $", "note"]
    return pd.DataFrame(results)[cols].style.format({"cost $": "{:.5f}", "seconds": "{:.1f}"}, na_rep="-")

def show(results, labels=None):
    # The answers, one after another.
    for i, r in enumerate(results):
        title = labels[i] if labels else f'{r["model"]}  [{r["tier"]}]'
        print("=" * 78)
        print(title)
        print("-" * 78)
        print(r["answer"] or r["note"])
    print("=" * 78)

def spent():
    print(f"This session so far: {len(LOG)} requests, ${sum(r['cost $'] for r in LOG):.4f}")

print("Helpers ready:", ", ".join(["ask", "compare", "receipts", "show", "spent"]))

## 4. Experiment A — an easy question, to every rung

One prompt goes to all nine models at the same moment. The first table is the receipts; the answers follow. Frontier models may take up to a minute.

In [ ]:
prompt_a = "In two sentences, explain what an API is to a diplomat who has never written code."

results_a = compare(prompt_a)
display(receipts(results_a))
show(results_a)
spent()

**Checkpoint 4 — record evidence, not a vibe.**

- Which answers would you be content to use? Which is the cheapest of those?
- Quote one specific phrase where a tiny model's answer differs from a frontier model's.
- Look at *seconds* and *of which thinking*. Some models spend tokens reasoning before they answer, and you pay for those tokens. Which ones did?

If a row shows `ERROR` or `no visible answer`, that is data too: note which model and what the message said. A small model served by a single host is more likely to be busy when a whole class calls it at once.

## 5. Experiment B — questions with a right answer

Opinions about writing quality are hard to compare. So here are four small questions where Python can compute the correct answer and check each model against it. None requires knowledge; each requires care.

A language model does not calculate or count the way a program does. It predicts likely text. Watch where that difference shows.

In [ ]:
start_date = datetime.date(2025, 11, 17)
phrase = "extraterritorial jurisdiction"
a, b = 4817, 6293

def words(text):
    return re.sub(r"[^a-z0-9 ]", " ", text.lower()).split()

def last_number(text):
    found = re.findall(r"\d[\d,]*", text)
    return int(found[-1].replace(",", "")) if found and found[-1].replace(",", "") else None

due = start_date + datetime.timedelta(days=90)

QUIZ = [
    {"name": "date + 90 days",
     "prompt": f"What calendar date is exactly 90 days after {start_date.day} {start_date:%B %Y}? "
               "Reply with the date only, written like 3 March 2026.",
     "truth": f"{due.day} {due:%B %Y}",
     "check": lambda ans: {str(due.day), f"{due:%B}".lower(), str(due.year)} <= set(words(ans))},
    {"name": "multiply",
     "prompt": f"What is {a} x {b}? Reply with digits only.",
     "truth": a * b,
     "check": lambda ans: last_number(ans) == a * b},
    {"name": "count a letter",
     "prompt": f'How many times does the letter "r" appear in the phrase "{phrase}"? Reply with digits only.',
     "truth": phrase.count("r"),
     "check": lambda ans: last_number(ans) == phrase.count("r")},
    {"name": "exactly 12 words",
     "prompt": "Write one sentence about trade sanctions that is exactly twelve words long. Reply with the sentence only.",
     "truth": "any 12-word sentence",
     "check": lambda ans: len(ans.replace("*", "").split()) == 12},
]

for q in QUIZ:
    print(f'{q["name"]:>18}:  correct answer = {q["truth"]}')

jobs = [(m, q) for q in QUIZ for m in LADDER]
with ThreadPoolExecutor(max_workers=8) as pool:
    quiz_results = list(pool.map(lambda job: ask(job[0], job[1]["prompt"]), jobs))

board = {m: {"tier": LADDER[m]} for m in LADDER}
for (m, q), r in zip(jobs, quiz_results):
    board[m][q["name"]] = "?" if r["note"].startswith("ERROR") else ("yes" if q["check"](r["answer"]) else "NO")
for m in LADDER:
    board[m]["score"] = f'{sum(v == "yes" for v in board[m].values())} / {len(QUIZ)}'
    board[m]["cost $"] = round(sum(r["cost $"] for (mm, _), r in zip(jobs, quiz_results) if mm == m), 5)

display(pd.DataFrame(board).T)
spent()

In [ ]:
# Read what the models actually wrote for one question. Change the number to 0, 1, 2 or 3.
which = 0

q = QUIZ[which]
print(q["prompt"], "\nCorrect answer:", q["truth"], "\n")
for (m, qq), r in zip(jobs, quiz_results):
    if qq is q:
        print(f'{m:<36} {(r["answer"] or r["note"])[:110]!r}')

**Checkpoint 5.**

- At which rung does each question start to fail? Is it the same rung for all four?
- Did any expensive model miss one? Did any cheap one get all four?
- The checker is a few lines of code and can be wrong. Use the cell above to find one answer you think it graded unfairly, in either direction.
- Four questions is a very small test. What would you want to see before telling a colleague "model X can do arithmetic"?

## 6. Experiment C — a blind taste test

Now a task closer to your own work, where there is no single right answer. Five models, one from each tier, write a short briefing. You will see the answers labeled A to E **without** the model names.

In [ ]:
prompt_c = (
    "A city of 400,000 is deciding whether its housing office should use an AI model to draft replies "
    "to residents' questions about rent assistance. In exactly three bullet points, give the strongest "
    "argument for, the strongest argument against, and one safeguard you would require. Under 90 words."
)

blind_models = [m for m in ["anthropic/claude-fable-5.1", "anthropic/claude-sonnet-5.5", "openai/gpt-6-luna",
                            "google/gemma-4-31b-it", "meta-llama/llama-3.2-1b-instruct"] if m in LADDER]
results_c = compare(prompt_c, blind_models)
random.shuffle(results_c)
show(results_c, labels=[f"Answer {chr(65 + i)}" for i in range(len(results_c))])

**Checkpoint 6 — rank before you reveal.** Order the answers from best to worst and write one reason for your top and bottom choices. Then guess which answer came from the most expensive model.

> My ranking, best to worst:
> Why the top one:
> Why the bottom one:
> My guess for the most expensive model:

Only then run the next cell.

In [ ]:
reveal = pd.DataFrame(results_c)[["model", "tier", "seconds", "tokens out", "cost $"]]
reveal.insert(0, "answer", [chr(65 + i) for i in range(len(results_c))])
display(reveal.style.format({"cost $": "{:.5f}", "seconds": "{:.1f}"}, na_rep="-"))

Compare your ranking with your partner's before you compare it with the price list. Where you disagree with each other, what were you each rewarding?

## 7. Experiment D — the same model, asked three times

Before concluding that model X beats model Y, check how much one model varies against itself. Last week you saw why: at each step the model samples from a probability distribution over next tokens. The same prompt is sent three times to one cheap model and one tiny model.

In [ ]:
prompt_d = "Suggest a name for a new international treaty on AI safety. Reply with the name only."

for m in ["openai/gpt-6-luna", "meta-llama/llama-3.2-1b-instruct"]:
    if m in CATALOG:
        print(m)
        for r in compare(prompt_d, [m, m, m]):
            print("   ", r["answer"] or r["note"])

**Checkpoint 7.** Rerun Experiment A or C if you like and see whether your judgment of any model changes. What does this imply for a claim like "I tried it and the model said …"?

## 8. Experiment E — the bill at scale

One request costs a fraction of a cent on any rung, which makes the price ladder easy to ignore. It stops being ignorable when the same request runs thousands of times. This cell makes no API calls; it multiplies list prices.

The starting scenario: an agency receives 10,000 public comments on a proposed rule, each about 1,500 tokens (roughly 1,100 words), and wants a 200-token summary of each. Change the three numbers to a scenario from your own field.

In [ ]:
documents = 10_000
tokens_in_each = 1_500
tokens_out_each = 200

rows = []
for m in LADDER:
    total = documents * (tokens_in_each * price(m, "prompt") + tokens_out_each * price(m, "completion")) / 1_000_000
    rows.append({"tier": LADDER[m], "model": m, "total $": total})
display(pd.DataFrame(rows).style.format({"total $": "{:,.2f}"}))
print("Models that think before answering bill those thinking tokens as output, so their real total is higher.")

**Checkpoint 8.** Given what you saw in Experiments A to C, which rung would you choose for this job, and what would you check before trusting it with 10,000 documents? Who can afford the top rung for this task, and who cannot?

## 9. Your turn — a prompt from your own project

Write one prompt that matters for your term project, and choose at least three models from different tiers. `EXTRAS` (section 3) lists more IDs worth trying, including models from xAI, Meta, Alibaba, Zhipu and Moonshot; any ID from [openrouter.ai/models](https://openrouter.ai/models) works.

Keep to the frontier tier for one or two runs, and do the rest of your experimenting on the cheaper rungs.

In [ ]:
my_prompt = "Replace this with your own prompt."

my_models = [
    "anthropic/claude-sonnet-5.5",
    "openai/gpt-6-luna",
    "mistralai/ministral-3b-2512",
]

my_results = compare(my_prompt, my_models)
display(receipts(my_results))
show(my_results)
spent()

## 10. Optional — the cheap shortlist, and a model that returns numbers

For routine experimenting, these five inexpensive models cover the main kinds of input and output. Everything so far has been text in, text out; that is one row of this table.

| Model ID | Input | Output | Good first question |
|---|---|---|---|
| `deepseek/deepseek-v4.1-flash` | text, image | text | Can it answer a short prompt? |
| `openai/gpt-6-luna` | text, image, file | text | How does another text model respond? |
| `openai/text-embedding-3-small` | text | vector | Which sentences are similar? |
| `google/gemini-3.1-flash-lite-image` | text, image | text, image | How can a model produce an image? |
| `openai/gpt-audio-mini` | text, audio | text, audio | How does an app send and receive sound? |

An **embedding model** returns a list of numbers, a vector, that places a text in a mathematical space. Texts used in similar ways get vectors pointing in similar directions, which we measure with **cosine similarity** (near 1 means similar). This is how search and retrieval tools decide which documents are "about" your question.

Replace the three sentences with: one of your own, a paraphrase that changes most of the words, and a sentence on a different topic that shares a word with the first. Predict which pair will score higher.

In [ ]:
texts = [
    "The ceasefire talks resumed in Geneva on Monday.",
    "Negotiators returned to Switzerland to discuss ending the fighting.",
    "The Geneva watch fair opened on Monday.",
]

emb = requests.post(f"{BASE_URL}/embeddings", headers=HEADERS, timeout=60,
                    json={"model": "openai/text-embedding-3-small", "input": texts}).json()
vectors = [np.array(item["embedding"]) for item in emb["data"]]

def cosine_similarity(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

print("Each text became a list of", len(vectors[0]), "numbers.")
print("sentence 1 vs 2:", round(cosine_similarity(vectors[0], vectors[1]), 3))
print("sentence 1 vs 3:", round(cosine_similarity(vectors[0], vectors[2]), 3))

## Wrap-up

Run the cell below to see what the session cost, then record in your homework log:

1. the ladder table from section 3 (a screenshot is fine);
2. your Checkpoint 3 predictions and whether Experiments A and B bore them out;
3. your blind ranking from Experiment C, written before the reveal, and what the reveal showed;
4. one answer you would not have trusted, with the model that gave it; and
5. your own prompt from section 9, the models you chose, and which you would use again.

When you are finished, use **Runtime → Disconnect and delete runtime**. The key stays in Colab's secret manager, not in the notebook.

In [ ]:
spent()
after = requests.get(f"{BASE_URL}/key", headers=HEADERS, timeout=30).json()["data"]
print("Key usage before: $", START_USAGE, "   now: $", after.get("usage"), "   remaining: $", after.get("limit_remaining"), sep="")